# VDJdb_valid: AIRR-format эпитопы + True-Negative клонотипы

Этот ноутбук:
1. Читает сырой VDJdb (TSV).
2. Находит топ-20 эпитопов отдельно для TRA и TRB (по числу уникальных клонотипов).
3. Приводит записи к минимальному AIRR-like формату: `clone_id, junction_aa, v_call, j_call, locus`.
4. Для каждого эпитопа добавляет true-negative (TN) клонотипы: 10% и 25% (сэмплируются из VDJdb той же цепи после исключения текущего эпитопа).
   TN дописываются в конец файла и получают `clone_id`, начиная с 2000.
5. Делает укороченные background-файлы: первые 100 000 строк для `tra_background.tsv` и `trb_background.tsv`.

Результаты сохраняются в: `vdjdb_valid/airr_format/`.


In [1]:
# =========================
# params
# =========================
from pathlib import Path

# 1) Сырой VDJdb (tsv)
RAW_VDJDB_TSV = Path("/projects/immunestatus/vdjdb/vdjdb-2025-07-30/vdjdb_full_filtered.txt")

# 2) (опционально) директория, где уже лежат background файлы в AIRR-формате
AIRR_FORMAT_DIR = Path("/projects/immunestatus/vdjdb/airr_format")  # где лежат tra_background.tsv / trb_background.tsv

# 3) куда писать результаты
OUT_ROOT = Path("/projects/immunestatus/vdjdb_olga")
OUT_AIRR = OUT_ROOT / "airr_format"
OUT_AIRR.mkdir(parents=True, exist_ok=True)

TOP_N_EPITOPES = 20
TN_PCTS = [0.10, 0.25, 0.5, 0.75]     # 10% и 25%
TN_CLONE_ID_START = 200000   # начиная с 2000

TN_CLONE_ID_START_OLGA = 200000    # Olga-версия: TN clone_id начинаются с 1000
BG_SLICE_START = 0            # Olga-версия: берем background iloc[200:1000]
BG_SLICE_STOP = 100000

RANDOM_SEED = 42


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

rng = np.random.default_rng(RANDOM_SEED)


tra_bg_in = AIRR_FORMAT_DIR / "tra_background.tsv"
trb_bg_in = AIRR_FORMAT_DIR / "trb_background.tsv"


## Загрузка VDJdb и авто-детект колонок

Ноутбук старается автоматически найти нужные поля (цепь, эпитоп, CDR3aa, V, J).
Если авто-детект не сработает — ниже есть `detected_cols`, и можно вручную поправить переменные.

In [3]:
raw = pd.read_csv(RAW_VDJDB_TSV, sep="\t", low_memory=False)
raw.columns = [c.replace(".", "_") for c in raw.columns]  # на всякий случай
raw.head()

,cdr3_alpha,v_alpha,j_alpha,cdr3_beta,v_beta,d_beta,j_beta,species,mhc_a,mhc_b,mhc_class,antigen_epitope,antigen_gene,antigen_species,reference_id,method_identification,method_frequency,method_singlecell,method_sequencing,method_verification,meta_study_id,meta_cell_subset,meta_subject_cohort,meta_subject_id,meta_replica_id,meta_clone_id,meta_epitope_id,meta_tissue,meta_donor_MHC,meta_donor_MHC_method,meta_structure_id,cdr3fix_alpha,cdr3fix_beta,vdjdb_score
0,NaN,NaN,NaN,CASSIVGGNEQFF,TRBV19*01,NaN,TRBJ2-1*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,tetramer-sort,70.5%,NaN,amplicon-seq,"antigen-loaded-targets,antigen-expressing-targets",NaN,CD8+,MA-EBV-SN,donor1,NaN,NaN,M158–66,PBMC,"A02,A03,B07,B27,C01,C07",NaN,NaN,NaN,"{'cdr3': 'CASSIVGGNEQFF', 'cdr3_old': 'CASSIVG...",3
1,NaN,NaN,NaN,CASSMRSTGELFF,TRBV19*01,NaN,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,tetramer-sort,11.0%,NaN,amplicon-seq,"antigen-loaded-targets,antigen-expressing-targets",NaN,CD8+,MA-EBV-SN,donor1,NaN,NaN,M158–66,PBMC,"A02,A03,B07,B27,C01,C07",NaN,NaN,NaN,"{'cdr3': 'CASSMRSTGELFF', 'cdr3_old': 'CASSMRS...",3
2,NaN,NaN,NaN,CASSIRSAWAQYF,TRBV19*01,NaN,TRBJ2-3*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,tetramer-sort,1.9%,NaN,amplicon-seq,"antigen-loaded-targets,antigen-expressing-targets",NaN,CD8+,MA-EBV-SN IAV-M1,donor1,NaN,NaN,M158–66,PBMC,"A02,A03,B07,B27,C01,C07",NaN,NaN,NaN,"{'cdr3': 'CASSIRSAWAQYF', 'cdr3_old': 'CASSIRS...",2
3,NaN,NaN,NaN,CASSQRSTGELFF,TRBV19*01,NaN,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,tetramer-sort,1.7%,NaN,amplicon-seq,"antigen-loaded-targets,antigen-expressing-targets",NaN,CD8+,MA-EBV-SN IAV-M1,donor1,NaN,NaN,M158–66,PBMC,"A02,A03,B07,B27,C01,C07",NaN,NaN,NaN,"{'cdr3': 'CASSQRSTGELFF', 'cdr3_old': 'CASSQRS...",2
4,NaN,NaN,NaN,CASSIRSSYEQYF,TRBV19*01,NaN,TRBJ2-7*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,tetramer-sort,0.9%,NaN,amplicon-seq,"antigen-loaded-targets,antigen-expressing-targets",NaN,CD8+,MA-EBV-SN IAV-M1,donor1,NaN,NaN,M158–66,PBMC,"A02,A03,B07,B27,C01,C07",NaN,NaN,NaN,"{'cdr3': 'CASSIRSSYEQYF', 'cdr3_old': 'CASSIRS...",3


In [4]:
VDJDB_COLS = {
    "epitope": "antigen_epitope",

    "TRA": {"cdr3": "cdr3_alpha", "v": "v_alpha", "j": "j_alpha"},
    "TRB": {"cdr3": "cdr3_beta",  "v": "v_beta",  "j": "j_beta"},
}

In [5]:
def to_airr_like(df: pd.DataFrame, chain: str) -> pd.DataFrame:
    """
    chain: 'TRA' or 'TRB'
    """
    chain = chain.upper()
    assert chain in ("TRA", "TRB")

    cols = VDJDB_COLS[chain]
    ep_col = VDJDB_COLS["epitope"]

    # фильтр: берем только строки где нужная цепь реально есть
    m = df[cols["cdr3"]].notna()
    sub = df.loc[m, [cols["cdr3"], cols["v"], cols["j"], ep_col]].copy()

    # минимальный AIRR-like
    out = pd.DataFrame({
        "clone_id": sub.index,
        "junction_aa": sub[cols["cdr3"]].astype(str),
        "v_call": sub[cols["v"]].astype(str),
        "j_call": sub[cols["j"]].astype(str),
        "locus": "alpha" if chain == "TRA" else "beta",
        "antigen_epitope": sub[ep_col].astype(str),
    })

    # при желании можно выкинуть мусорные "nan" строки после astype(str)
    out = out[out["junction_aa"].str.lower() != "nan"].copy()

    return out

## Выбор топ-20 эпитопов по TRA и TRB (по числу уникальных клонотипов)

In [6]:
def top_epitopes(df: pd.DataFrame, chain: str, n: int = 20) -> list[str]:
    chain = chain.upper()
    cols = VDJDB_COLS[chain]
    ep_col = VDJDB_COLS["epitope"]

    m = df[cols["cdr3"]].notna()
    sub = df.loc[m, [ep_col, cols["cdr3"], cols["v"], cols["j"]]].copy()

    sub["key"] = (
        sub[cols["cdr3"]].astype(str) + "|" +
        sub[cols["v"]].astype(str) + "|" +
        sub[cols["j"]].astype(str)
    )

    counts = sub.groupby(ep_col)["key"].nunique().sort_values(ascending=False)
    return counts.head(n).index.tolist()

top_tra = top_epitopes(raw, "TRA", n=20)
top_trb = top_epitopes(raw, "TRB", n=20)


## OLGA data for sampling

In [7]:
def get_olga(in_path: Path, n: int = 100_000, start: int = 200_000):
    if not in_path.exists():
        raise FileNotFoundError(in_path)
    chunk = pd.read_csv(in_path, sep="\t", nrows=start+n).loc[start: start+n]
    return chunk

tra_tn_candidates = get_olga(tra_bg_in)
trb_tn_candidates = get_olga(trb_bg_in)


In [8]:
tra_tn_candidates

,junction_aa,v_call,j_call,locus
200000,CAVDGSGGSYIPTF,TRAV39,TRAJ6,alpha
200001,CALSEPSGNTPLVF,TRAV9-2,TRAJ29,alpha
200002,CAFMKLRSNDYKLSF,TRAV38-1,TRAJ20,alpha
200003,CAMSTGSNDYKLSF,TRAV12-3,TRAJ20,alpha
200004,CAMISGYNKLIF,TRAV12-3,TRAJ4,alpha
...,...,...,...,...
299995,CVVNLVGTYKYIF,TRAV12-1,TRAJ40,alpha
299996,CAASLYIPTF,TRAV13-1,TRAJ6,alpha
299997,CAAFRGSNYKLTF,TRAV29/DV5,TRAJ53,alpha
299998,CALKGATNKLIF,TRAV9-2,TRAJ32,alpha


## Функции экспорта AIRR-like и генерации TN

In [9]:
def sample_true_negative(
    chain: str,
    n_pos: int,
    frac: float,
    seed: int = 1,
) -> pd.DataFrame:
    if chain.upper() == 'TRA':
        cand_airr = tra_tn_candidates
    else:
        cand_airr = trb_tn_candidates
    
    # сколько TN надо
    n_tn = int(round(frac * n_pos))
    n_tn = min(n_tn, len(cand_airr))
    tn = cand_airr.sample(n=n_tn, random_state=seed).copy()
    tn["clone_id"] = np.arange(TN_CLONE_ID_START, TN_CLONE_ID_START + len(tn), dtype=int)
    tn['antigen_epitope'] = 'olga'

    return tn

## Генерация файлов по эпитопам (TRA/TRB) + TN10/TN25

Для каждого эпитопа сохраняем 4 файла:
- `*_tn10.tsv` и `*_tn25.tsv` — полный
- `*_uni_tn10.tsv` и `*_uni_tn25.tsv` — уникализированный

Плюс файлы меток: `*_labels.tsv`.

In [10]:
from pathlib import Path

OUTDIR = Path("/projects/immunestatus/vdjdb_olga/airr_format")
OUTDIR.mkdir(parents=True, exist_ok=True)

def build_and_save_for_epitope(df_raw: pd.DataFrame, chain: str, epitope: str, seed: int = 1):
    chain = chain.upper()
    ep_col = VDJDB_COLS["epitope"]

    # positive: только этот эпитоп + нужная цепь присутствует
    pos_raw = df_raw[df_raw[ep_col] == epitope].copy()
    pos_airr = to_airr_like(pos_raw, chain)

    # куда складывать метаданные/пути
    prefix = "tra" if chain == "TRA" else "trb"
    stats = {
        "epitope": epitope,
        "chain": chain,
        "n_pos": len(pos_airr),
    }

    # для каждого процента TN: pos + tn(frac)
    for frac in TN_PCTS:
        tn = sample_true_negative(
            chain,
            n_pos=len(pos_airr),
            frac=frac,
            seed=seed,
        )

        out = pd.concat([pos_airr, tn])

        # tn10 / tn25 / tn50 / tn75
        tn_tag = f"tn{int(round(frac * 100))}"
        out_path = OUTDIR / f"{prefix}_vdjdb_{epitope}_{tn_tag}.tsv"
        out.to_csv(out_path, sep="\t", index=False)

        stats[f"n_{tn_tag}"] = len(tn)
        stats[f"path_{tn_tag}"] = str(out_path)

    return stats

In [11]:
summary = []

for ep in top_tra:
    summary.append(build_and_save_for_epitope(raw, "TRA", ep, seed=1))

for ep in top_trb:
    summary.append(build_and_save_for_epitope(raw, "TRB", ep, seed=1))

summary_df = pd.DataFrame(summary)
summary_df

,epitope,chain,n_pos,n_tn10,path_tn10,n_tn25,path_tn25,n_tn50,path_tn50,n_tn75,path_tn75
0,KLGGALQAK,TRA,13726,1373,/projects/immunestatus/vdjdb_olga/airr_format/...,3432,/projects/immunestatus/vdjdb_olga/airr_format/...,6863,/projects/immunestatus/vdjdb_olga/airr_format/...,10294,/projects/immunestatus/vdjdb_olga/airr_format/...
1,GILGFVFTL,TRA,8481,848,/projects/immunestatus/vdjdb_olga/airr_format/...,2120,/projects/immunestatus/vdjdb_olga/airr_format/...,4240,/projects/immunestatus/vdjdb_olga/airr_format/...,6361,/projects/immunestatus/vdjdb_olga/airr_format/...
2,NLVPMVATV,TRA,2596,260,/projects/immunestatus/vdjdb_olga/airr_format/...,649,/projects/immunestatus/vdjdb_olga/airr_format/...,1298,/projects/immunestatus/vdjdb_olga/airr_format/...,1947,/projects/immunestatus/vdjdb_olga/airr_format/...
3,AVFDRKSDAK,TRA,1731,173,/projects/immunestatus/vdjdb_olga/airr_format/...,433,/projects/immunestatus/vdjdb_olga/airr_format/...,866,/projects/immunestatus/vdjdb_olga/airr_format/...,1298,/projects/immunestatus/vdjdb_olga/airr_format/...
4,RAKFKQLL,TRA,2182,218,/projects/immunestatus/vdjdb_olga/airr_format/...,546,/projects/immunestatus/vdjdb_olga/airr_format/...,1091,/projects/immunestatus/vdjdb_olga/airr_format/...,1636,/projects/immunestatus/vdjdb_olga/airr_format/...
5,YLQPRTFLL,TRA,849,85,/projects/immunestatus/vdjdb_olga/airr_format/...,212,/projects/immunestatus/vdjdb_olga/airr_format/...,424,/projects/immunestatus/vdjdb_olga/airr_format/...,637,/projects/immunestatus/vdjdb_olga/airr_format/...
6,IVTDFSVIK,TRA,714,71,/projects/immunestatus/vdjdb_olga/airr_format/...,178,/projects/immunestatus/vdjdb_olga/airr_format/...,357,/projects/immunestatus/vdjdb_olga/airr_format/...,536,/projects/immunestatus/vdjdb_olga/airr_format/...
7,LLAGIGTVPI,TRA,643,64,/projects/immunestatus/vdjdb_olga/airr_format/...,161,/projects/immunestatus/vdjdb_olga/airr_format/...,322,/projects/immunestatus/vdjdb_olga/airr_format/...,482,/projects/immunestatus/vdjdb_olga/airr_format/...
8,RLRAEAQVK,TRA,424,42,/projects/immunestatus/vdjdb_olga/airr_format/...,106,/projects/immunestatus/vdjdb_olga/airr_format/...,212,/projects/immunestatus/vdjdb_olga/airr_format/...,318,/projects/immunestatus/vdjdb_olga/airr_format/...
9,GLCTLVAML,TRA,1002,100,/projects/immunestatus/vdjdb_olga/airr_format/...,250,/projects/immunestatus/vdjdb_olga/airr_format/...,501,/projects/immunestatus/vdjdb_olga/airr_format/...,752,/projects/immunestatus/vdjdb_olga/airr_format/...


## Background: первые 100 000 строк

In [12]:
def head_100k(in_path: Path, out_path: Path, n: int = 100_000):
    if not in_path.exists():
        raise FileNotFoundError(in_path)
    chunk = pd.read_csv(in_path, sep="\t", nrows=n)
    chunk.to_csv(out_path, sep="\t", index=False)
    return len(chunk)

tra_bg_in = AIRR_FORMAT_DIR / "tra_background.tsv"
trb_bg_in = AIRR_FORMAT_DIR / "trb_background.tsv"

tra_bg_out = OUT_AIRR / "tra_background_100k.tsv"
trb_bg_out = OUT_AIRR / "trb_background_100k.tsv"

n1 = head_100k(tra_bg_in, tra_bg_out, n=100_000)
n2 = head_100k(trb_bg_in, trb_bg_out, n=100_000)

n1, n2, tra_bg_out, trb_bg_out


(100000,
 100000,
 PosixPath('/projects/immunestatus/vdjdb_olga/airr_format/tra_background_100k.tsv'),
 PosixPath('/projects/immunestatus/vdjdb_olga/airr_format/trb_background_100k.tsv'))

## Быстрая проверка результатов

In [13]:
created = sorted([p.name for p in OUT_AIRR.glob("*.tsv")])
print("TSV files:", len(created))
created[:40]


TSV files: 162


['tra_background_100k.tsv',
 'tra_vdjdb_ASNENMETM_tn10.tsv',
 'tra_vdjdb_ASNENMETM_tn25.tsv',
 'tra_vdjdb_ASNENMETM_tn50.tsv',
 'tra_vdjdb_ASNENMETM_tn75.tsv',
 'tra_vdjdb_AVFDRKSDAK_tn10.tsv',
 'tra_vdjdb_AVFDRKSDAK_tn25.tsv',
 'tra_vdjdb_AVFDRKSDAK_tn50.tsv',
 'tra_vdjdb_AVFDRKSDAK_tn75.tsv',
 'tra_vdjdb_CINGVCWTV_tn10.tsv',
 'tra_vdjdb_CINGVCWTV_tn25.tsv',
 'tra_vdjdb_CINGVCWTV_tn50.tsv',
 'tra_vdjdb_CINGVCWTV_tn75.tsv',
 'tra_vdjdb_ELAGIGILTV_tn10.tsv',
 'tra_vdjdb_ELAGIGILTV_tn25.tsv',
 'tra_vdjdb_ELAGIGILTV_tn50.tsv',
 'tra_vdjdb_ELAGIGILTV_tn75.tsv',
 'tra_vdjdb_GILGFVFTL_tn10.tsv',
 'tra_vdjdb_GILGFVFTL_tn25.tsv',
 'tra_vdjdb_GILGFVFTL_tn50.tsv',
 'tra_vdjdb_GILGFVFTL_tn75.tsv',
 'tra_vdjdb_GLCTLVAML_tn10.tsv',
 'tra_vdjdb_GLCTLVAML_tn25.tsv',
 'tra_vdjdb_GLCTLVAML_tn50.tsv',
 'tra_vdjdb_GLCTLVAML_tn75.tsv',
 'tra_vdjdb_IVTDFSVIK_tn10.tsv',
 'tra_vdjdb_IVTDFSVIK_tn25.tsv',
 'tra_vdjdb_IVTDFSVIK_tn50.tsv',
 'tra_vdjdb_IVTDFSVIK_tn75.tsv',
 'tra_vdjdb_KLGGALQAK_tn10.tsv',
 'tra_v

In [14]:
# sanity-check: открыть один файл и убедиться, что TN в конце и clone_id с 2000
example = next(iter(OUT_AIRR.glob("trb_vdjdb_*_tn10.tsv")))
ex = pd.read_csv(example, sep="\t")
print("Example:", example.name)
display(ex.head(3))
display(ex.tail(3))
print("min clone_id:", ex["clone_id"].min(), "max clone_id:", ex["clone_id"].max())
print("TN rows (clone_id>=200000):", (ex["clone_id"]>=TN_CLONE_ID_START).sum())


Example: trb_vdjdb_RLRAEAQVK_tn10.tsv


,clone_id,junction_aa,v_call,j_call,locus,antigen_epitope
0,36673,CASGGKVFPPYEQYF,TRBV12-5*01,TRBJ2-7*01,beta,RLRAEAQVK
1,36893,CASREIQGSGANVLTF,TRBV7-9*01,TRBJ2-6*01,beta,RLRAEAQVK
2,36962,CASSQVGRPYNEQFF,TRBV4-3*01,TRBJ2-1*01,beta,RLRAEAQVK


,clone_id,junction_aa,v_call,j_call,locus,antigen_epitope
463,200039,CASSPGTEEETQYF,TRBV18,TRBJ2-5,beta,olga
464,200040,CASSLKLAGVTDTQYF,TRBV7-6,TRBJ2-3,beta,olga
465,200041,CSAARFGGMDEKLFF,TRBV20-1,TRBJ1-4,beta,olga


min clone_id: 36673 max clone_id: 200041
TN rows (clone_id>=200000): 42
